# Trisha v2 — Advanced Models + Ensemble
RTX 3050 6GB | Improved Augmentations | Focal Loss | Oversample | TTA


In [1]:
import sys
sys.path.insert(0, "../..")

from modules.utils.config import *
from modules.utils.seed import set_seed
from modules.data.dataset import get_dataloaders_v2, get_test_loader
from modules.models.factory import TrashClassifier
from modules.training.train import fit
from modules.training.loss import FocalLoss

set_seed(SEED)
print(f"Device: {DEVICE}")


d:\Kuliah\LOMBA\Satria-Data\big-data-big-trouble\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda


In [2]:
MODELS = [
    "convnext_base",
    "swin_small_patch4_window7_224",
    "tf_efficientnetv2_m",
]

BATCH_SIZE = 16
ACCUMULATION_STEPS = 2
NUM_WORKERS = 2
EPOCHS_HEAD = 8
EPOCHS_FINETUNE = 15
LR_HEAD = 1e-3
LR_FINETUNE = 1e-4
PATIENCE = 8


In [3]:
train_loader, val_loader, val_ds = get_dataloaders_v2(
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    oversample=True,
)
print(f"Train: {len(train_loader.dataset)} | Val: {len(val_loader.dataset)}")


Train: 21219 | Val: 5305


d:\Kuliah\LOMBA\Satria-Data\big-data-big-trouble\notebooks\experiments\../..\modules\data\dataset.py:176: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:209.)
  weights = 1.0 / class_counts[train_labels].float()


In [4]:
results = []


## **Convnext_base**

In [ ]:
criterion = FocalLoss(alpha=val_ds.class_weights.to(DEVICE), gamma=2.0)

model = TrashClassifier("convnext_base", num_classes=NUM_CLASSES).to(DEVICE)
res = fit(
    model, train_loader, val_loader,
    name="trisha_v2_convnext_base",
    encoder_name="convnext_base",
    accumulation_steps=ACCUMULATION_STEPS,
    epochs_head=EPOCHS_HEAD,
    epochs_finetune=EPOCHS_FINETUNE,
    lr_head=LR_HEAD,
    lr_finetune=LR_FINETUNE,
    patience=PATIENCE,
    class_weights=val_ds.class_weights,
    criterion=criterion,
    device=DEVICE,
)
results.append(res)
print(f">>> convnext_base: val_f1_macro = {res['best_val_f1']:.4f}")


d:\Kuliah\LOMBA\Satria-Data\big-data-big-trouble\.venv\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Acer\.cache\huggingface\hub\models--timm--convnext_base.fb_in22k_ft_in1k. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)



=== trisha_v2_convnext_base: Phase 1 — Head Only ===


  E01: train_loss=0.0552  val_f1=0.9433  best=0.9433


trisha_v2_convnext_base train:   0%|          | 0/1327 [00:00<?, ?it/s]

In [ ]:
criterion = FocalLoss(alpha=val_ds.class_weights.to(DEVICE), gamma=2.0)

model = TrashClassifier("swin_small_patch4_window7_224", num_classes=NUM_CLASSES).to(DEVICE)
res = fit(
    model, train_loader, val_loader,
    name="trisha_v2_swin_small_patch4_window7_224",
    encoder_name="swin_small_patch4_window7_224",
    accumulation_steps=ACCUMULATION_STEPS,
    epochs_head=EPOCHS_HEAD,
    epochs_finetune=EPOCHS_FINETUNE,
    lr_head=LR_HEAD,
    lr_finetune=LR_FINETUNE,
    patience=PATIENCE,
    class_weights=val_ds.class_weights,
    criterion=criterion,
    device=DEVICE,
)
results.append(res)
print(f">>> swin_small_patch4_window7_224: val_f1_macro = {res['best_val_f1']:.4f}")


In [ ]:
criterion = FocalLoss(alpha=val_ds.class_weights.to(DEVICE), gamma=2.0)

model = TrashClassifier("tf_efficientnetv2_m", num_classes=NUM_CLASSES).to(DEVICE)
res = fit(
    model, train_loader, val_loader,
    name="trisha_v2_tf_efficientnetv2_m",
    encoder_name="tf_efficientnetv2_m",
    accumulation_steps=ACCUMULATION_STEPS,
    epochs_head=EPOCHS_HEAD,
    epochs_finetune=EPOCHS_FINETUNE,
    lr_head=LR_HEAD,
    lr_finetune=LR_FINETUNE,
    patience=PATIENCE,
    class_weights=val_ds.class_weights,
    criterion=criterion,
    device=DEVICE,
)
results.append(res)
print(f">>> tf_efficientnetv2_m: val_f1_macro = {res['best_val_f1']:.4f}")


In [ ]:
import json
import pandas as pd

records = []
for f in sorted(RESULTS.glob("trisha_v2_*.json")):
    with open(f) as fh:
        records.append(json.load(fh))

summary = pd.DataFrame(records)
summary = summary.sort_values("best_val_f1", ascending=False)
summary[["name", "best_val_f1", "f1_per_class"]].to_csv(RESULTS / "trisha_v2_summary.csv", index=False)
summary[["name", "best_val_f1", "f1_per_class"]]


In [ ]:
import torch
import torch.nn.functional as F
from tqdm import tqdm
from modules.ensemble.vote import generate_submission
from modules.utils.config import CLASS_LABELS

json_files = sorted(RESULTS.glob("trisha_v2_*.json"))
print(f"Loading {len(json_files)} models for ensemble...")

records = []
for f in json_files:
    with open(f) as fh:
        records.append(json.load(fh))

records.sort(key=lambda r: r["best_val_f1"], reverse=True)

models = []
for rec in records:
    m = TrashClassifier(rec["encoder_name"], num_classes=NUM_CLASSES).to(DEVICE)
    pt_path = RESULTS / f"{rec['name']}.pt"
    m.load_state_dict(torch.load(pt_path, map_location=DEVICE))
    m.eval()
    models.append(m)
    print(f"  {rec['name']}: f1={rec['best_val_f1']:.4f}")

BATCH_SIZE_TEST = 64
test_loader = get_test_loader(batch_size=BATCH_SIZE_TEST, num_workers=NUM_WORKERS)

@torch.inference_mode()
def predict_with_tta(models, loader, device=DEVICE):
    all_probs = []
    for inputs, _ in tqdm(loader, desc="TTA Ensemble"):
        inputs = inputs.to(device)
        flipped = torch.flip(inputs, dims=[-1])
        logits_orig = torch.stack([m(inputs) for m in models])
        logits_flip = torch.stack([m(flipped) for m in models])
        probs_orig = F.softmax(logits_orig, dim=-1)
        probs_flip = F.softmax(logits_flip, dim=-1)
        probs = (probs_orig + probs_flip) / 2
        all_probs.append(probs.mean(dim=0).cpu())
    return torch.cat(all_probs)

pred_probs = predict_with_tta(models, test_loader, device=DEVICE)
generate_submission(pred_probs, output_path=RESULTS / "trisha_v2_submission.csv")
